In [78]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name =  "green-trips"

In [79]:
from model import Ride, ride_deserializer

In [80]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database-2',
    value_deserializer=ride_deserializer
)

In [81]:
record = next(consumer)

In [82]:
record.value

Ride(PULocationID=247, DOLocationID=69, trip_distance=0.7, total_amount=10.0, lpep_pickup_datetime=1759278107000, lpep_dropoff_datetime=1759278277000, passenger_count=1.0, tip_amount=1.7)

In [83]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='secretpassword'
)

conn.autocommit = True
cur = conn.cursor()

In [84]:
drop_table_sql = """
DROP TABLE IF EXISTS green_trips;
"""

cur.execute(drop_table_sql)
conn.commit()

print("Old table green_trips deleted successfully")

Old table green_trips deleted successfully


In [85]:

create_table_sql = """
CREATE TABLE IF NOT EXISTS green_trips (
    lpep_pickup_datetime   TEXT,
    lpep_dropoff_datetime  TEXT,
    PULocationID           BIGINT,
    DOLocationID           BIGINT,
    passenger_count        BIGINT,
    trip_distance          FLOAT,
    tip_amount             FLOAT,
    total_amount           FLOAT
);
"""

cur.execute(create_table_sql)
conn.commit()
print("Table green_trips created successfully")


Table green_trips created successfully


In [86]:
view = """(
SELECT * FROM green_trips LIMIT 5
);
"""

cur.execute(view)
conn.commit()
print("View created successfully")

View created successfully


In [87]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value

    # Convert timestamps from milliseconds to datetime
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)

    try:
        cur.execute(
            """INSERT INTO green_trips
               (lpep_pickup_datetime, lpep_dropoff_datetime,
                PULocationID, DOLocationID,
                passenger_count, trip_distance,
                tip_amount, total_amount)
               VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
            (
                pickup_dt,
                dropoff_dt,
                int(ride.PULocationID),
                int(ride.DOLocationID),
                int(ride.passenger_count),
                int(ride.trip_distance),
                int(ride.tip_amount),
                int(ride.total_amount)
            )
        )
    except Exception as e:
        print(f"Error inserting row: {e}")
        print(f"Data: {ride}")
        continue

    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...


Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
Inserted 4100 rows...
Inserted 4200 rows...
Inserted 4300 rows...
Inserted 4400 rows...
Inserted 4500 rows...
Inserted 4600 rows...
Inserted 4700 rows...
Inserted 4800 row

KeyboardInterrupt: 

In [88]:
count_sql = "SELECT COUNT(*) FROM green_trips;"

cur.execute(count_sql)
row_count = cur.fetchone()[0]

print(f"Number of rows in green_trips: {row_count}")

Number of rows in green_trips: 44401


In [89]:
count_sql = """
SELECT COUNT(*)
FROM green_trips
WHERE trip_distance >= 5;
"""

cur.execute(count_sql)
count_over_5 = cur.fetchone()[0]

print(f"Number of trips with trip_distance > 5: {count_over_5}")

Number of trips with trip_distance > 5: 5914


In [ ]:
cur.execute("SELECT * FROM green_trips LIMIT 5;")
rows = cur.fetchall()
for row in rows:
    print(row)
else:
    print("Table 'processed_events' does not exist.")